In [ ]:
#to-do list

#allow hits to be those that don't ionise but where the rt matches that in the platemap

In [1]:
import sys
sys.path.insert(1, 'OneDrive/Documents/GitHub/PyParse')
import pandas as pd
import math
from rdkit import Chem
from rdkit.Chem import AllChem
from rdkit.Chem import Descriptors
from rdkit.Chem import Draw
from rdkit.Chem import PandasTools
from statistics import mean
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import seaborn as sns

In [2]:
def getMSData(spectrum):
    """
    Takes the specific region of the rpt file pertaining to 
    m/z data for a specific peak in a specific well, and 
    returns a 2-D list containing all m/z peaks and their 
    normalised intensity.
    
    :param spectrum: Section of rpt file as string
    
    :return: 2-D list in following format: 
        [m/z value, normalised intensity of that value]
    """
    
    masses = []
    total = 0
    
    lineData = spectrum.split(";Mass\t% BPI")[1].split("\n")
    for line in lineData[1:]:
        if line == "}": #stop the for loop if end of MSData section is reached
            break
    
        massData = line.split("\t")
        if len(massData) == 2:
            floatData = [float(i) for i in massData] #convert all data to float
            masses.append(floatData)
            
            total = total + floatData[1]
    #Remove any masses which, as a percentage, round to 0 
    #to remove unnecessary baseline ions
    refined_masses = []
    for i in masses:
        if math.floor((i[1]/total)*100) > 0:
            refined_masses.append([i[0], i[1]])
    
    return refined_masses
   
def getUVData(spectrum, min_uv_threshold):
    """
    Takes the specific region of the rpt file pertaining to 
    UV absorbance spectrum data for a specific peak in a specific
    well, and returns a list containing all the maxima of that spectrum.
    The height of the maxima must be greater than the min_uv_threshold
    specified in options.
    
    :param spectrum: Section of rpt file as string
    
    :return: UV maxima as a list
    """

    UVmaxima = []
    
    lineData = spectrum.split(";Mass\t% BPI")[1].split("\n")
    UVx = []
    UVy = []
    for line in lineData:
        
        if line == "}":#stop for loop if end of UVData section is reached
            break
        
        UVdata = line.split("\t")
        if len(UVdata) == 2:
            UVx.append(float(UVdata[0]))
            UVy.append(abs(float(UVdata[1])))
    if UVy[0] > UVy[1] and UVy[0] > min_uv_threshold:
        UVmaxima.append(UVx[0])
    for i in range(1, len(UVy)-1):
        if UVy[i] > UVy[i-1] and UVy[i] > UVy[i+1] and UVy[i] > min_uv_threshold:
            UVmaxima.append(UVx[i])
    if UVy[-1] > UVy[-2] and UVy[-1] > min_uv_threshold:
        UVmaxima.append(UVx[-1])

        
    return UVmaxima

def getUserReadableWell(wellno, plate_col_no):
    """
    Converts the well as a number into a user-friendly string,
    e.g. well 11 becomes "B5" for a 4*6 well plate

    :param wellno: An integer representing a specific well on the plate
    
    :return: A string representing a specific well on the plate
    """
    
    rowVal = math.floor((wellno-1) / plate_col_no)
    colVal = (wellno) % plate_col_no
    if colVal == 0:
        colVal = plate_col_no
    
    label = f'{chr(ord("@")+(rowVal)+1)}{colVal}'
    return label

In [317]:
class rawData:
    def __init__(self, inputfile, row_no = 0, col_no = 0):
        #self.rawDADTable = pd.DataFrame(columns =['well', 'peakID', 'time', 'area', 'areaAbs', 'pStart', 'pEnd'])
        #self.rawUVTable = pd.DataFrame(columns =['well', 'peakID', 'time', 'UVvalue'])
        #self.rawMSTable = pd.DataFrame(columns =['well', 'peakID', 'time', 'MSvalue', 'MSintensity', "MStype"])
        #self.rawELSDTable = pd.DataFrame(columns =['well', 'peakID', 'time', 'area', 'areaAbs', 'pStart', 'pEnd'])
        #self.rawTraceTable = pd.DataFrame(columns =['well', 'time', 'height'])
        self.row_no = row_no
        self.col_no = col_no
        self.sample_IDs = {}
        
        with open(inputfile, errors = "ignore") as f:
            fullText = f.read()
            self.wellData = fullText.split("[SAMPLE]")[1:] #Split the file into individual wells
    
    def getWellFormat(self):
        #Get the plate dimensions from the first sample
        #Data sample: #Plate	01TL,XY,SD,1: 8,2:12,3: 90.0...
        #where number of rows is 8 and number of columns in 12
        #Overwrite the default option, but ensure that the user retains control
        #such that empty rows can be removed from the heatmap. 
        if self.row_no == 0 or self.col_no == 0:
            self.row_no = int(self.wellData[0].split("\n")[17].split(",")[3].split(":")[1])
            self.col_no = int(self.wellData[0].split("\n")[17].split(",")[4].split(":")[1])
            self.plate_cols_for_extraction = self.col_no
        else:
            self.plate_cols_for_extraction = int(self.wellData[0].split("\n")[17].split(",")[4].split(":")[1])
            
            
        #Find from rpt file how each well is specified 
        #Data sample: #Plate	01TL,XY,SD,1: 8,2:12,3: 90.0...
        self.row_col_order = self.wellData[0].split("\n")[17].split(",")[1]
        self.well_type = self.wellData[0].split("\n")[17].split(",")[2]
        
    def getWell(self, position):
        well_number = -1
        #If the well type is just a Single Digit...
        if self.well_type == "SD":
            #If the well is simply an integer between 1 and infinity
            #Single line function to trim full string to just the well number used
            well_number = int(self.wellData[position].split("Well")[1].split("\n")[0].split(":")[1].strip()) 
            #If the column number specified by the user is different to that found in the rpt file, 
            #this is the result of the user looking to trim off blank columns. Only wells described by 
            #a single digit need to be modified to take this into account. 
            if self.col_no != self.plate_cols_for_extraction:
                well_number = math.floor(well_number / self.plate_cols_for_extraction)*self.col_no + (well_number % self.plate_cols_for_extraction)

        #If the well type is a combination of letters/numbers...
        else:
            #Find if the well  
            if self.row_col_order == "XY":
                column = self.wellData[position].split("Well")[1].split("\n")[0].split(":")[1].split(",")[0].strip()
                row = self.wellData[position].split("Well")[1].split("\n")[0].split(":")[1].split(",")[1].strip()
            else: 
                column = self.wellData[position].split("Well")[1].split("\n")[0].split(":")[1].split(",")[1].strip()
                row = self.wellData[position].split("Well")[1].split("\n")[0].split(":")[1].split(",")[0].strip()

            #Convert the column to integer, either by direct
            #conversion, or by finding position in the alphabet
            try:
                col_as_int = int(column)
            except:
                col_as_int = ord(column.capitalize()) - 64

            #Convert the row to integer, either by direct
            #conversion, or by finding position in the alphabet
            try: 
                row_as_int = int(row)
            except: 
                row_as_int = ord(row.capitalize()) - 64

            #Calculate the wellno as a single integer
            well_number = (row_as_int - 1) * self.plate_cols_for_extraction + col_as_int
        return well_number
    
    def processDAD(self):
        peak_list = []
        for i in range(len(self.wellData)):
            functions = self.wellData[i].split("[FUNCTION]")
            well_number = self.getWell(i)
            
            #Make sure the sample ID has been added for this well. This process is repeated for the other 
            #"process_something" functions, so that the data is available regardless of which detectors 
            #are of interest. 
            if well_number not in self.sample_IDs:
                well_ID = functions[0].split("SampleID")[1].split("\n")[0].strip()  
                self.sample_IDs[well_number] = well_ID
                
            for j in range(len(functions[1:])):
                function = functions[1:][j]
                lines = function.split("\n")
                
                #get peakarea for this peak
                if "Type\tDAD" in lines[4]:
                    chromatograms = function.split("[CHROMATOGRAM]")[1:]
                    for chromatogram in chromatograms:
                        c_lines = chromatogram.split("\n")
                        if "Description\tDAD:" in c_lines[3]:
                            spectra = function.split("[SPECTRUM]")[1:] #split by spectrum (i.e. each peak)
                            #chroma[wellno] = getChromatogram(chromatogram.split("[TRACE]")[1]) #get chromatogram for this well
                            peaks = chromatogram.split("[PEAK]")[1:]
                            for peak in peaks:
                                new_entry = {
                                    "well": well_number,
                                    "peakID": int(peak.split("Peak ID")[1].split("\n")[0].strip()),
                                    "time": float(peak.split("Time")[1].split("\n")[0].strip()),
                                    "pStart": peak.split("Peak\t")[1].split("\n")[0].split("\t")[0],
                                    "pEnd": peak.split("Peak\t")[1].split("\n")[0].split("\t")[1],
                                    "area": float(peak.split("Area %Total")[1].split("\n")[0].strip()),
                                    "areaAbs": float(peak.split("AreaAbs")[1].split("\n")[0].strip())
                                }
                                peak_list.append(new_entry)

        self.rawDADTable = pd.DataFrame(peak_list)
        
        
    def processUV(self, min_uv_threshold = 20): 
        peak_list = []
        for i in range(len(self.wellData)):
            functions = self.wellData[i].split("[FUNCTION]")
            well_number = self.getWell(i)
            
            #Make sure the sample ID has been added for this well. This process is repeated for the other 
            #"process_something" functions, so that the data is available regardless of which detectors 
            #are of interest. 
            if well_number not in self.sample_IDs:
                well_ID = functions[0].split("SampleID")[1].split("\n")[0].strip()  
                self.sample_IDs[well_number] = well_ID
                
            for j in range(len(functions[1:])):
                function = functions[1:][j]
                lines = function.split("\n")
                if "Type\tDAD" in lines[4]:
                    spectra = function.split("[SPECTRUM]")[1:] #split by spectrum (i.e. each peak)

                    for spectrum in spectra:
                        UVdata = getUVData(spectrum, min_uv_threshold)
                        for maxima in UVdata:
                            new_entry = {
                                "well": well_number,
                                "peakID": int(spectrum.split("Peak ID")[1].split("\n")[0].strip()),
                                "time": float(spectrum.split("Time")[1].split("\n")[0].strip()),
                                "UVvalue": maxima,
                            }
                            peak_list.append(new_entry)

        if len(peak_list) == 0:
            self.rawUVTable = pd.DataFrame(columns =['well', 'peakID', 'time', 'UVvalue'])
        else:
            self.rawUVTable = pd.DataFrame(peak_list)
        
        
    def processMS(self):
        peak_list = []
        for i in range(len(self.wellData)):
            functions = self.wellData[i].split("[FUNCTION]")
            well_number = self.getWell(i)
            
            #Make sure the sample ID has been added for this well. This process is repeated for the other 
            #"process_something" functions, so that the data is available regardless of which detectors 
            #are of interest. 
            if well_number not in self.sample_IDs:
                well_ID = functions[0].split("SampleID")[1].split("\n")[0].strip()  
                self.sample_IDs[well_number] = well_ID
                
            for j in range(len(functions[1:])):
                function = functions[1:][j]
                lines = function.split("\n")
                if "IonMode\tES" in lines[3]:
                    
                    spectra = function.split("[SPECTRUM]")[1:] #split by spectrum (i.e. each peak)
                    for spectrum in spectra:
                        if "IonMode\tES+" in lines[3]:
                            MStype = "+"
                        elif "IonMode\tES-" in lines[3]:
                            MStype = "-"

                        MSdata = getMSData(spectrum)
                        for ion in MSdata:
                            new_entry = {
                                "well": well_number,
                                "peakID": int(spectrum.split("Peak ID")[1].split("\n")[0].strip()),
                                "time": float(spectrum.split("Time")[1].split("\n")[0].strip()),
                                "MSvalue": ion[0],
                                "MSintensity": ion[1],
                                "MStype": MStype
                            }
                            peak_list.append(new_entry)

        
        self.rawMSTable = pd.DataFrame(peak_list)
        
        #Calculate the sum of the MSintensities in each peak, then calculate the of each MSintensity to this total
        total_intensities = self.rawMSTable.groupby(["well", "peakID", "MStype"]).agg(total_intensity=('MSintensity', 'sum'))
        self.rawMSTable = self.rawMSTable.join(total_intensities, on=["well", "peakID", "MStype"], rsuffix = "right")
        self.rawMSTable["perc_intensity"] = 100 * self.rawMSTable["MSintensity"] / self.rawMSTable["total_intensity"]
        
    def processTrace(self, points_per_trace = 500):
        
        """
        Takes the region of text in rpt file corresponding to the chromatogram,
        and extracts the data into two lists, corresponding to x-values and y-values. 

        :param spectrum: Section of rpt file as string

        :return: A list of 2 lists, [x-values, y-values]
        """            
        trace_list = []
        
        for i in range(len(self.wellData)):
            functions = self.wellData[i].split("[FUNCTION]")
            well_number = self.getWell(i)
            
            for function in functions[1:]:
                lines = function.split("\n")
                if "Type\tDAD" in lines[4]:
                    chromatograms = function.split("[CHROMATOGRAM]")[1:]
                    for chromatogram in chromatograms:
                        c_lines = chromatogram.split("\n")
                        if "Description\tDAD:" in c_lines[3]:
                            trace_text = chromatogram.split("[TRACE]")[1].split("\n")

                            length = len(trace_text)
                            n_val = math.ceil(length / points_per_trace)

                            for index, value in enumerate(trace_text[1:]):
                                if value == "}": #stop for loop if end of data section is reached
                                    break
                                elif index % n_val == 0:

                                    if value != "{":
                                        data = value.split("\t")
                                        goingin = {
                                            "well": well_number,
                                            "time": float(data[0]),
                                            "height": float(data[1])
                                        }
                                        trace_list.append(goingin)

        self.rawTraceTable = pd.DataFrame(trace_list)
        
        
            

In [318]:
inputfile = "example_dataset/Waters/Example1/example_rpt.rpt"
#inputfile = "example_dataset/Waters/Example2/LC-MS Data for 48-Well Plate.rpt"
#inputfile = "C:/Users/joe.mason/OneDrive - Domainex/Desktop/test.rpt"
test = rawData(inputfile)

In [319]:
test.getWellFormat()

In [320]:
test.processDAD()
test.processMS()
test.processUV()
test.processTrace()

In [321]:
test.rawTraceTable

,well,time,height
0,1,0.01417,0.002
1,1,0.01875,0.004
2,1,0.02333,0.005
3,1,0.02792,0.008
4,1,0.03250,0.010
...,...,...,...
10459,24,1.98955,-4.285
10460,24,1.99413,-4.285
10461,24,1.99871,-4.285
10462,24,2.00330,-4.285


In [8]:
test.rawDADTable

,well,peakID,time,pStart,pEnd,area,areaAbs
0,1,1,1.2300,1.2154,1.2525,100.00,1.071910e+06
1,2,1,0.3533,0.3400,0.3700,1.07,1.085240e+04
2,2,2,1.2958,1.2817,1.3192,98.93,1.001665e+06
3,3,1,0.5863,0.5758,0.6071,100.00,1.190068e+06
4,4,1,0.5483,0.5383,0.5692,98.21,1.185495e+06
5,4,2,1.1246,1.1109,1.1467,1.79,2.156398e+04
6,5,1,1.2246,1.2104,1.2484,97.78,6.553948e+05
7,5,2,1.3688,1.3546,1.3929,2.22,1.486370e+04
8,6,1,1.1142,1.1000,1.1379,100.00,1.121448e+06
9,7,1,1.2284,1.2138,1.2542,100.00,9.312498e+05


In [477]:
class Assignment:
    def __init__(self, filename, plate_col_no):   
        #read csv file into dataframe
        #replace empty cells with an empty string
        #convert all column names to lower case and remove whitespace
        self.inputCSV = pd.read_csv(filename)
        self.inputCSV.fillna("", inplace=True)
        self.inputCSV.columns = self.inputCSV.columns.str.strip().str.lower()
        
        self.plate_col_no = plate_col_no
        
        
    
    #Fn to convert a well name like B5 to machine format (11)
    def convertWellToNum(self, wells):
        result = []
        
        for well in wells:
            row = well[0]
            column = well[1:]
            #if the format of the row/column conforms to expectations
            if isinstance(int(column[0]), int):
                result.append(int((ord(row) - 65) * self.plate_col_no + int(column)))
            #Plates with more than 26 rows are unsupported at present. 
            else:
                logging.info("The well specified implies an unsupported plate.")
                sys.exit(2)
        return result
    
    #Fn to convert smiles into canonicalised smiles
    def getCanonSmiles(self, smiles):
        mol = Chem.MolFromSmiles(smiles.strip())
        return Chem.MolToSmiles(mol)
            
    def generateCPTable(self):
        compound_list = []
        type_dic = {
            "desired product smiles": "Product",
            "limiting reactant smiles":"Reactant",
            "internalstd smiles": "InternalSTD"
        }
        counter = {
            "desired product smiles": 1,
            "limiting reactant smiles": 1,
            "byproduct": 1
        }
        
        #Canonicalise all the incoming smiles in key columns in case the user hadn't done so already
        for col in self.inputCSV:
            if col in type_dic or ("byproduct" in col and "smiles" in col):
                self.inputCSV[col] = self.inputCSV[col].apply(self.getCanonSmiles)
        
        for col in self.inputCSV:
            if col in type_dic or ("byproduct" in col and "smiles" in col):
                cpname_column = f'{col.split(" smiles")[0]} name'
                cprt_column = f'{col.split(" smiles")[0]} rt'
                
                #group the input CSV by canonical smiles in that column, aggregated the wells into a list
                groupeddf = self.inputCSV.groupby(col, as_index=False)[["well"]].agg(lambda x: list(x))
                
                #Iterate through each of those grouped entries
                for index, row in groupeddf.iterrows():
                    cpindex = row[col]
                    cptype = type_dic[col] if col in type_dic else "byproduct"
                    #get a name for the compound if one was provided
                    name = ""
                    if cpname_column in self.inputCSV.columns:
                        name_series = self.inputCSV.loc[self.inputCSV[col] == row[col]][cpname_column]
                        potential_names = [x for x in name_series if x != ""]
                        if len(potential_names) != 0:
                            name = potential_names[0]
                    #if a name could not be generated, create a generic one using a simple counter
                    if name == "":
                        if col == "internalstd smiles":
                            name = "InternalSTD"
                        elif col in counter:
                            name = f'{type_dic[col]}{counter[col]}'
                            counter[col] = counter[col] + 1
                        elif "byproduct" in col:
                            name = f'Byproduct{counter["byproduct"]}'
                            counter["byproduct"] = counter["byproduct"] + 1
                    
                    #get a rentention time for the compound if one was provided
                    rt = 0
                    if cprt_column in self.inputCSV.columns:
                        rt_series = self.inputCSV.loc[self.inputCSV[col] == row[col]][cprt_column]
                        potential_rt = [x for x in rt_series if x != ""]
                        if len(potential_rt) != 0:
                            rt = potential_rt[0]
                            
                    new_entry = {
                        "smiles": cpindex,
                        "type": cptype,
                        "locations": row["well"],
                        "name": name,
                        "rt": rt,
                        "comments": []
                    }
                    compound_list.append(new_entry)
        
        self.cpTable = pd.DataFrame(compound_list)
        
        #set the index to be the canonicalised smiles
        self.cpTable.index = list(self.cpTable["smiles"])
        
        #convert the "A1" style well IDs into a integer, to allow matching to a well in the rawData
        self.cpTable["locations"] = self.cpTable["locations"].apply(self.convertWellToNum)
    

    
    def generateEMs(self, calc_boc):
        
        def getMW(smiles):
            mol = Chem.MolFromSmiles(smiles)
            return round(Descriptors.ExactMolWt(mol), 2)
        
        def transform_and_getMW(smiles, smirks, stage):
            try:
                mol = Chem.MolFromSmiles(smiles)
                rxn1 = AllChem.ReactionFromSmarts(smirks)
                new_mol1 = rxn1.RunReactants((mol, ))[0][0]
                #Sanitise the molecule to make sure that a sensible molecule was produced. 
                Chem.SanitizeMol(new_mol1)
                return round(Descriptors.ExactMolWt(new_mol1), 2)
            except:
                if ("Cl" in smiles or "Br" in smiles) and stage == "mass2":
                    mol = Chem.MolFromSmiles(smiles)
                    return round(Descriptors.ExactMolWt(mol), 2) + 2
                else:
                    return 0
            
        self.cpTable["mass1"] = self.cpTable["smiles"].apply(lambda smiles: getMW(smiles))
        
        if calc_boc == "True":
            
            smirks1 = "[NX3,n:1][C:2](=[O:3])[O:4][C]([CH3])([CH3])[CH3]>>[*:1][C:2](=[O:3])[O:4]"
            self.cpTable["mass2"] = self.cpTable["smiles"].apply(lambda smiles: transform_and_getMW(smiles, smirks1, "mass2"))
            
            smirks2 = "[NX3,n:1][C](=[O])[O][C]([CH3])([CH3])[CH3]>>[*:1][H]"
            self.cpTable["mass3"] = self.cpTable["smiles"].apply(lambda smiles: transform_and_getMW(smiles, smirks2, "mass3"))
            

    def findHits(self, msData, mass_abs_tol = 0.5, min_massconf_threshold = 10, 
                                      time_abs_tol = 0.025, calc_higherions = "True"):
        
        def getMatches(compound):
            
            def getMassWithHighestConf(row):

                MS_plus = list(df.loc[(df.index.isin(MS_hits)) & (df["MStype"] == "+") &
                                       (df["MSintensity"] == row["max_intensity"]), "MSvalue"])
                MS_minus = list(df.loc[(df.index.isin(MS_hits)) & (df["MStype"] == "-") &
                                       (df["MSintensity"] == row["max_intensity"]), "MSvalue"])
                row["MS_plus"] = MS_plus[0] if len(MS_plus) > 0 else "-"
                row["MS_minus"] = MS_minus[0] if len(MS_minus) > 0 else "-"
                
                return row
            
            df = msData.loc[msData["well"].isin(compound["locations"])]
            
            MS_hits = []
            
            for mass in [compound["mass1"], compound["mass2"], compound["mass3"]]:
                hits = df.loc[(df["MStype"] == "+") & 
                                   ((abs(df["MSvalue"] - (mass + 1.01)) <= mass_abs_tol) |
                                    (abs(df["MSvalue"] - (mass + 2.02)/2) <= mass_abs_tol) |
                                    (abs(df["MSvalue"] - (mass + 3.03)/3) <= mass_abs_tol))]
                MS_hits = MS_hits + list(hits.index.values)
                
                hits = df.loc[(df["MStype"] == "-") & 
                                   (abs(df["MSvalue"] - (mass - 1.01)) <= mass_abs_tol)]
                MS_hits = MS_hits + list(hits.index.values)
            
            grouped = (df[df.index.isin(MS_hits)].groupby(["well", "peakID"], as_index = False)
                        .agg(mass_conf = ("perc_intensity", "sum"), max_intensity = ("MSintensity", "max")))
            grouped = grouped.apply(getMassWithHighestConf, axis = 1)
           
            grouped = grouped.loc[grouped["mass_conf"] >= min_massconf_threshold]
            grouped = grouped.drop("max_intensity", axis=1)

            return grouped.to_dict("records")                      

        self.cpTable["hits"] = self.cpTable.apply(getMatches, axis = 1)
    
    def validateHits(self, lcData, msData, uvData, time_abs_tol = 0.025, massconf_threshold = 0.5, uv_abs_tol = 10,
                    uv_cluster_threshold = 0.5, uv_match_threshold = 0.5, cluster_size_threshold = 0.8, min_no_of_wells = 5,
                    validate = "True", mass_or_area = "mass_conf"):
        
        def getRelevantPeaks(x, data):
            
            return list(data[(data["well"] == x["well"]) & (data["peakID"] == x["peakID"])].index.values)
            
        def clusterHits(compound):
                
            relevant_indexes = []
            for hit in compound["hits"]:
                relevant_indexes = relevant_indexes + getRelevantPeaks(hit, lcData)
            
            df = lcData[lcData.index.isin(relevant_indexes)]
            df.sort_values("time", inplace = True)
            
            clusters = []
            for index in df.index:
                if len(clusters) == 0:
                    clusters.append([index])
                else:
                    clusterFound = False
                    for cluster in clusters:
                        mean_rt = mean([df.loc[i, "time"] for i in cluster])

                        if abs(mean_rt - df.loc[index, "time"]) < time_abs_tol:
                            cluster.append(index)
                            clusterFound = True
                            break
                    if not clusterFound:
                        clusters.append([index])
            
            #At this point, clusters contains the indexes of the relevant rows of lcData
            return clusters
        
        def getClusterBand(compound):
            clusterbands = []
            for cluster in compound["clusters"]:
                mean = lcData.loc[lcData.index.isin(cluster)]["time"].mean()
                clusterbands.append(round(mean, 5))
            return clusterbands
                       
        def selectCluster_ifrt(row):
            comments = row["comments"]
            #If the user has specified a retention time, we should select only the cluster 
            #that is closest to that retention time, and within the specified time_abs_tol
            if row["rt"] != 0:
                suitable_clusters = [index for index, i in enumerate(row["cluster_bands"]) if abs(i - row["rt"]) < time_abs_tol]

                #If there is more than one cluster close to the specified retention time
                #take only the cluster which is closest 
                if len(suitable_clusters) > 1:
                    diffs = [row["cluster_bands"][index]-row["rt"] for index in suitable_clusters]
                    index_min = min(range(len(diffs)), key=diffs.__getitem__)
                    row["clusters"] = [row["clusters"][suitable_clusters[index_min]]]
                    #Update the cluster bands to only include the correct label
                    cluster_bands = [row["cluster_bands"][suitable_clusters[index_min]]]

                    row["comments"].append("<strong>Multiple clusters were found close the specified"
                                        " retention time.</strong>")
                    row["comments"].append(f'<strong>Cluster {index_min} was selected as it was closest'
                                        ' to the specified retention time.</strong>')
                elif len(suitable_clusters) == 1:
                    row["clusters"] = [row["clusters"][suitable_clusters[0]]]
                    row["cluster_bands"] = [row["cluster_bands"][suitable_clusters[0]]]
                    row["comments"].append("<strong>A single cluster was found close the specified"
                                        " retention time and this was selected for analysis.</strong>")
                else:
                    row["comments"].append("<strong>No cluster was found near to the specified "
                                        "retention time. Proceeding with analysis using all "
                                        f'{len(row["clusters"])} clusters.</strong>')
            return row
            
        def refineClustersByTime(row):
            """
            Takes in input cluster of all the hit peaks, 
            and refines them by finding a mid-value for the retention
            time based on which hit has the greatest number of nearest neighbours. 
            Sorts the best hits into "green", uncertain ones into "orange" 
            and those where another peak closer to the mid-value was found
            in the same well into "discarded". 

            :param cluster: list of dictionaries, where each dictionary is a hit
            :param comments: A list of comments for that structure so far.

            :return: List comprising [a dictionary for the refined cluster, list of comments]
            """

            [clusters, comments, expected_rt] = [row["clusters"], row["comments"], row["rt"]]
            refined_clusters = []

            for cluster in clusters:
                refined_cluster = {
                    "green":[],
                    "orange": [],
                    "discarded": [],
                }

                mid_values = []
                mid_value = 0

                if expected_rt != 0:
                    mid_value = expected_rt
                else: 
                    for i in cluster:
                        mid_value = lcData.loc[i, "time"]
                        mid_values.append([mid_value, len([lcData.loc[j, "time"] for j in cluster 
                                                           if abs(lcData.loc[j, "time"] - mid_value) < time_abs_tol/4])])
                    mid_value = max(mid_values, key = lambda x: x[1])[0]

                #sort the peaks by the well they occupy
                peaks_by_wells = {}
                for i in cluster:
                    if lcData.loc[i, "well"] not in peaks_by_wells:
                        peaks_by_wells[lcData.loc[i, "well"]] = []
                    peaks_by_wells[lcData.loc[i, "well"]].append(i)

                #For each well, select the peak that's closest to the mid-value in cases
                #where there was more than one hit in that cluster in one well
                #Use peakAdded to ensure that only a single peak per well is added to green, in the 
                #unlikely case that there are two peaks in the same well with the same retention time
                #(i.e. LCMS machine processing error)
                for index, well in peaks_by_wells.items():
                    if len(well) > 1:
                        min_diff = min([abs(lcData.loc[i, "time"]-mid_value) for i in well])
                        peakAdded = False
                        for i in well:

                            if abs(lcData.loc[i, "time"]-mid_value) == min_diff and not peakAdded:
                                refined_cluster["green"].append(i)
                                peakAdded = True
                            else:
                                refined_cluster["discarded"].append(i)
                                row["comments"].append(f'Peak at {lcData.loc[i, "time"]} '
                                            f'in well {getUserReadableWell(index, self.plate_col_no)} was discarded '
                                            'as there was an alternative peak '
                                            'in the same well which was closer to the '
                                            'mid-point of the cluster.')
                    else:
                        refined_cluster["green"].append(well[0])

                #Refine these hits further by finding those which are within time_abs_tol/2
                #of the mid-value. Any others are marked as tentative and the user is alerted.         
                ref2_cluster = {
                    "green":[],
                    "orange": refined_cluster["orange"],
                    "discarded": refined_cluster["discarded"],
                }        
                
                for i in refined_cluster["green"]:
                    if abs(lcData.loc[i, "time"] - mid_value) < time_abs_tol / 2:
                        ref2_cluster["green"].append(i)
                    else:
                        ref2_cluster["orange"].append(i)
                        comments.append(f'Peak at {lcData.loc[i, "time"]} in '
                                       f'well {getUserReadableWell(lcData.loc[i, "well"], self.plate_col_no)} was '
                                       'marked as tentative as it was found to be too '
                                       'far from the mid-value of the cluster.')
                refined_clusters.append(ref2_cluster)

            row["clusters"] = refined_clusters
            return row
        
        def refineClustersByMassConf(row):
            """
            Takes in input cluster of all the hit peaks, 
            and refines them by ensuring all peaks have a similar mass confidence
            to the cluster's mean. Those which do are left in "green"; 
            those which don't are moved to the "orange" category.

            :param cluster: a dict, with list of dicts for each header
            :param comments: A list of comments for the compound so far

            :return: List comprising [a dictionary for the refined cluster, list of comments]
            """
            refined_clusters = []
            for cluster in row["clusters"]:
                refined_cluster = {
                    "green": [],
                    "orange": cluster["orange"],
                    "discarded": cluster["discarded"],
                }
                if len(cluster["green"]) > 0:
                    #Get a dictionary of the mass_conf for each peak, indexed by the index present in lcData
                    mass_conf_dict = {}
                    for i in cluster["green"]:
                        well = lcData.loc[i, "well"]
                        peakID = lcData.loc[i, "peakID"]
                        for j in row["hits"]:
                            if j["well"] == well and j["peakID"] == peakID:
                                mass_conf_dict[i] = j["mass_conf"]

                    #find what the mean mass confidence is of all peaks
                    #currently under the "green" category
                    total = sum(mass_conf_dict[x] for x in cluster["green"])
                    mean_mass_conf = total / len(cluster["green"])

                    for i in cluster["green"]:
                        if mass_conf_dict[i] < mean_mass_conf * massconf_threshold:
                            refined_cluster["orange"].append(i)
                            row["comments"].append(f'Peak at {lcData.loc[i, "time"]} in well '
                                            f'{getUserReadableWell(lcData.loc[i, "well"], self.plate_col_no)} '
                                            'was marked tentative due to poor mass '
                                            'confidence in comparison to the rest of the cluster.')
                        else:
                            refined_cluster["green"].append(i)
                refined_clusters.append(refined_cluster)
            row["clusters"] = refined_clusters
            return row
        
        def refineClustersByUV(row):
            """
            Takes in input cluster of all the hit peaks, 
            and refines the cluster by ensuring all peaks have a similar set
            of UV maxima. Those which do are left in "green", those which
            don't are moved to the "orange" category

            :param cluster: a dict, with list of dicts for each header
            :param UVdatafound: boolean for whether the rpt data contains UV data
            :param comments: A list of comments for that structure so far

            :return: List comprising [a dictionary for the refined cluster, list of comments]
            """
            refined_clusters = []
            
            for cluster in row["clusters"]:
                refined_cluster = {
                    "green": [],
                    "orange": cluster["orange"],
                    "discarded": cluster["discarded"],
                }

                if len(cluster["green"]) > 0:
                    

                    UVclusters = []

                    for i in cluster["green"]:
                        for UV in uvData.loc[(uvData["well"] == lcData.loc[i, "well"]) & (uvData["peakID"] == lcData.loc[i, "peakID"])]["UVvalue"]:
                            if len(UVclusters) == 0:
                                UVclusters.append([UV])

                            else:
                                clusterFound = False
                                for UVcluster in UVclusters:
                                    if abs(UVcluster[-1] - UV) < uv_abs_tol:
                                        UVcluster.append(UV)
                                        clusterFound = True
                                        break
                                if not clusterFound:
                                    UVclusters.append([UV])

                    meanUV = []
                    #Check to ensure UVclusters isn't an empty set
                    if len(UVclusters) > 0:
                        lengthOfMostCommon = max([len(UVcluster) for UVcluster in UVclusters])

                        for UVcluster in UVclusters:
                            if len(UVcluster) >= lengthOfMostCommon * uv_cluster_threshold:
                                meanUV.append(mean(UVcluster))

                        for i in cluster["green"]:
                            UVvalues = (uvData.loc[(uvData["well"] == lcData.loc[i, "well"]) 
                                               & (uvData["peakID"] == lcData.loc[i, "peakID"])]["UVvalue"].values)
                            intersection = []
                            for meanvalue in meanUV:
                                for UV in UVvalues:
                                    if abs(meanvalue - UV) < uv_abs_tol:
                                        intersection.append(meanvalue)

                            if len(intersection) >= len(meanUV) * uv_match_threshold:
                                refined_cluster["green"].append(i)
                            else:
                                refined_cluster["orange"].append(i)
                                row["comments"].append(f'Peak at {lcData.loc[i, "time"]} in well '
                                                f'{getUserReadableWell(lcData.loc[i, "time"], self.plate_col_no)} '
                                                'was marked tentative due to mismatch in UV maxima with the rest of the cluster.')
                    #If UVclusters is an empty set, pass through the original hits without further
                    #validation. 
                    else:
                        row["comments"].append('UV validation not permitted where cluster contains peaks with no UV data. '
                            'UV validation was not performed.')
                        refined_cluster["green"] = cluster["green"]
                else:
                    row["comments"].append('UV data was not found for the plate, so UV validation was not performed.')
                    refined_cluster["green"] = cluster["green"]
                refined_clusters.append(refined_cluster)
            
            row["clusters"] = refined_clusters
            return row
        
        def selectClusterByMassConf(row):
            """
            If more than one cluster was found for the compound, 
            this function is called to try to select a single cluster based on 
            which cluster has the highest mean massConf. If more than one cluster
            has a close-to-highest-mean massconf, take them all. 

            :param clusters: a list of dictionaries, with list of dictionaries for each header
            :return refined_clusters: a list of dictionaries, with list of dictionaries for each header

            :return discarded_clusters: a list of dictionaries, with list of dictionaries for each header
            """
            
            refined_clusters = []
            discarded_clusters = []

            means = []
            #find the mean massconf for each cluster
            for cluster in row["clusters"]:
                if len(cluster["green"]) > 0:
                    #Get a dictionary of the mass_conf for each peak, indexed by the index present in lcData
                    mass_conf_dict = {}
                    for i in cluster["green"]:
                        well = lcData.loc[i, "well"]
                        peakID = lcData.loc[i, "peakID"]
                        for j in row["hits"]:
                            if j["well"] == well and j["peakID"] == peakID:
                                mass_conf_dict[i] = j["mass_conf"]
                    
                    #Calculate the mean mass_conf of the cluster
                    mean_mass_conf = sum([mass_conf_dict[i] for i in cluster["green"]]) / len(cluster["green"])
                    means.append(mean_mass_conf)
                else:
                    means.append(0)
                    
            #find the maximum mean value to compare all clusters against
            max_mean = max(means)
            
            #Filter the clusters by those which have a mean mass confidence that is at least the specified
            #percentage of the maximum observed mean mass confidence (set by massconf_threshold, typically 80%)
            for i in range(len(means)):
                if max_mean * massconf_threshold < means[i] or (max_mean == 0 and means[i] == 0):
                    refined_clusters.append(row["clusters"][i])
                else:
                    discarded_clusters.append(row["clusters"][i])
                    
            row["refined_clusters"] = refined_clusters
            row["discarded_clusters"] = discarded_clusters
            return row

        def selectClusterBySize(row):
            """
            If more than one cluster was found for the compound, 
            this function is called to try to select a single cluster based on 
            which cluster is the largest. If more than one cluster
            has a close-to-largest size, take them all. 

            :param clusters: a list of dictionaries, with list of dictionaries for each header
            :return refined_clusters: a list of dictionaries, with list of dictionaries for each header

            :return discarded_clusters: a list of dictionaries, with list of dictionaries for each header
            """

            refined_clusters2 = []
            discarded_clusters = row["discarded_clusters"]

            lengths = [len(cluster["green"]) + len(cluster["orange"]) for cluster in row["refined_clusters"]]
            max_length = max(lengths)

            for cluster in row["refined_clusters"]:
                if len(cluster["green"]) + len(cluster["orange"]) > max_length * cluster_size_threshold:
                    refined_clusters2.append(cluster)
                else:
                    discarded_clusters.append(cluster)

            #If there is still more than one cluster, reset the process using only those peaks
            #that haven't been marked as suspicious (orange)
            if len(refined_clusters2) > 1:

                lengths = [len(cluster["green"]) for cluster in row["refined_clusters"]]
                max_length = max(lengths)

                #As long as at least one cluster had a green hit, filter the clusters by size
                if max_length != 0:
                    refined_clusters2 = []
                    discarded_clusters = row["discarded_clusters"]

                    for cluster in row["refined_clusters"]:
                        if len(cluster["green"]) > max_length * cluster_size_threshold:
                            refined_clusters2.append(cluster)
                        else:
                            discarded_clusters.append(cluster)
            row["refined_clusters"] = refined_clusters2
            row["discarded_clusters"] = discarded_clusters
            return row
        
        def indexClusterByWells(row):
            cluster_by_well = {}
            for cluster in row["refined_clusters"]:
                for i in cluster["green"]:
                    if lcData.loc[i, "well"] not in cluster_by_well:
                        cluster_by_well[lcData.loc[i, "well"]] = {
                                "green": [],
                                "orange": [],
                                "discarded": [],
                                }

                    cluster_by_well[lcData.loc[i, "well"]]["green"].append(i)

                for i in cluster["orange"]:
                    if lcData.loc[i, "well"] not in cluster_by_well:
                        cluster_by_well[lcData.loc[i, "well"]] = {
                                "green": [],
                                "orange": [],
                                "discarded": [],
                                }

                    cluster_by_well[lcData.loc[i, "well"]]["orange"].append(i)

                for i in cluster["discarded"]:
                    if lcData.loc[i, "well"] not in cluster_by_well:
                        cluster_by_well[lcData.loc[i, "well"]] = {
                                "green": [],
                                "orange": [],
                                "discarded": [],
                                }

                    cluster_by_well[lcData.loc[i, "well"]]["discarded"].append(i)
                    
            row["clusters_indexed_by_well"] = cluster_by_well
            return row
            
        def refine_and_select(row): 
            #If there are sufficient wells to perform refine and select a cluster, do so. 
            #Otherwise, simply mark all hits as "green" fill in the necessary table structure. 
            if len(row["hits"]) > min_no_of_wells and validate == "True":
                row = refineClustersByTime(row)
                row = refineClustersByMassConf(row)
                row = refineClustersByUV(row)
                row = selectClusterByMassConf(row)
                row = selectClusterBySize(row)
            else:
                refined_clusters = []
                for cluster in row["clusters"]:
                    refined_cluster = {
                        "green": [],
                        "orange": [], 
                        "discarded": []
                    }
                    for i in cluster:
                        refined_cluster["green"].append(i)
                    refined_clusters.append(refined_cluster)
                row["refined_clusters"] = refined_clusters
                row["discarded_clusters"] = []
                
                if validate != "True":
                    row["comments"].append("Validation was not performed as requested by the user.")
                else:
                    row["comments"].append(f'Validation was not performed for {row["name"]} as '
                                        'there were insufficient hits.')
            return row
        
        def finalResult(row):
            final_result = {
                "green": [],
                "discarded": []    
            }
            
            for well in row["clusters_indexed_by_well"].values():
                #Move all discarded peaks immediately to the analogous result in "final result"
                final_result["discarded"] += well["discarded"]
                
                #Depending on how many green or orange peaks were found in this well, proceed accordingly, 
                #where green peaks are selected in preference to orange ones. 
                if len(well["green"]) > 0:
                    if len(well["green"]) == 1:
                        final_result["green"].append(well["green"][0])
                    else:
                        
                        #Get a list of peaks that are deemed suitable, either those that have the largest mass confidence
                        #or, in the case where the peaks are selected purely by their area, a dummy list. 
                        #We're going 
                        if mass_or_area == "mass_conf":
                            #Get a dictionary of the mass_conf for each peak, indexed by the index present in lcData
                            mass_conf_dict = {}
                            for i in well["green"]:
                                peak_well = lcData.loc[i, "well"]
                                peakID = lcData.loc[i, "peakID"]
                                for j in row["hits"]:
                                    if j["well"] == peak_well and j["peakID"] == peakID:
                                        mass_conf_dict[i] = j["mass_conf"]
                                        
                            max_mass_conf = max(mass_conf_dict.values())
                                                               
                            suitable_indexes = [i for i in well["green"] if mass_conf_dict[i] == max_mass_conf]
                        else:
                            suitable_indexes = well["green"]
                            
                        sorted_list = sorted(well["green"], key = lambda x: lcData.loc[x, "area"], reverse = True)

                        peakAdded = False
                        for i in sorted_list:
                            if i in suitable_indexes and peakAdded == False:
                                final_result["green"].append(i)
                                row["comments"].append(f'Peak with largest {mass_or_area} selected in '
                                                       'preference to others available for well '
                                                       f'{getUserReadableWell(lcData.loc[i, "well"], self.plate_col_no)}.')
                                peakAdded = True
                            else:
                                final_result["discarded"].append(i)
                                row["comments"].append(f'The peak at {lcData.loc[i, "time"]} in well '
                                                       f' {getUserReadableWell(lcData.loc[i, "well"], self.plate_col_no)} '
                                                       f'was discarded as it had a smaller {mass_or_area} than an '
                                                       'otherwise equally likely alternative.')
                                                            
                    #If we had a green peak to select, we can safely move all the orange peaks into the discarded pile
                    if len(well["orange"]) > 0:
                        final_result["discarded"] += well["orange"]
                        for i in well["orange"]:
                            row["comments"].append(f'The peak at {lcData.loc[i, "time"]} in well '
                                        f'{getUserReadableWell(lcData.loc[i, "well"], self.plate_col_no)} '
                                        f'for {row["name"]} was discarded because a better match was found.')
                
                elif len(well["orange"]) == 1:
                    final_result["green"].append(well["orange"][0])
                    row["comments"].append(f'<strong>The tentative peak at {lcData.loc[i, "time"]} in well '
                                        f'{getUserReadableWell(lcData.loc[i, "well"], self.plate_col_no)} was '
                                        f'used as there was no better option. User should check this well.</strong>')
                elif len(well["orange"]) > 1:
                    #Get a list of peaks that are deemed suitable, either those that have the largest mass confidence
                    #or, in the case where the peaks are selected purely by their area, a dummy list. 
                    #We're going 
                    if mass_or_area == "mass_conf":
                        #Get a dictionary of the mass_conf for each peak, indexed by the index present in lcData
                        mass_conf_dict = {}
                        for i in well["orange"]:
                            peak_well = lcData.loc[i, "well"]
                            peakID = lcData.loc[i, "peakID"]
                            for j in row["hits"]:
                                if j["well"] == peak_well and j["peakID"] == peakID:
                                    mass_conf_dict[i] = j["mass_conf"]

                        max_mass_conf = max(mass_conf_dict.values())

                        suitable_indexes = [i for i in well["orange"] if mass_conf_dict[i] == max_mass_conf]
                    else:
                        suitable_indexes = well["orange"]

                    sorted_list = sorted(well["orange"], key = lambda x: lcData.loc[x, "area"], reverse = True)

                    peakAdded = False
                    for i in sorted_list:
                        if i in suitable_indexes and peakAdded == False:
                            final_result["green"].append(i)
                            row["comments"].append(f'Tentatively assigned peak with largest {mass_or_area} selected in '
                                                   'preference to other tentative assignments available for well '
                                                   f'{getUserReadableWell(lcData.loc[i, "well"], self.plate_col_no)}.')
                            peakAdded = True
                        else:
                            final_result["discarded"].append(i)
                            row["comments"].append(f'<strong>The peak at {lcData.loc[i, "time"]} in well '
                                                   f' {getUserReadableWell(lcData.loc[i, "well"], self.plate_col_no)} '
                                                   f'was discarded as it had a smaller {mass_or_area} than an '
                                                   'otherwise equally likely alternative.</strong>')
                    
            row["final_result"] = final_result
            return row
        

                
            
        self.cpTable["clusters"] = self.cpTable.apply(clusterHits, axis = 1)
        self.cpTable["cluster_bands"] = self.cpTable.apply(getClusterBand, axis = 1)
        self.cpTable = self.cpTable.apply(selectCluster_ifrt, axis = 1)
        
        self.cpTable = self.cpTable.apply(refine_and_select, axis = 1)
        
        self.cpTable = self.cpTable.apply(indexClusterByWells, axis = 1)
        
        self.cpTable = self.cpTable.apply(finalResult, axis = 1)
        
    def setBestWell(self, row, outputTable, plot_type):
        
        if row["type"] == "Product":
            
            #Filter to the rows where the compound is found, then get a list of wells that have the 
            #maximum observed value for the given plot type (note: not guaranteed to return a single well!)
            new_slice = outputTable.loc[outputTable["well_no"].isin(row["locations"])]
            best_well_poss = list(new_slice.loc[(new_slice[plot_type] == new_slice[plot_type].max()) & 
                                 (new_slice[plot_type] > 0),  "well_no"])

            if len(best_well_poss) > 0:
                best_well = best_well_poss[0]
            
            #If no "best well" is found (perhaps because the internal standard wasn't found in that well, 
            #set the best well to be the one where the highest Parea was observed. 
            else:
                best_well_poss2 = list(new_slice.loc[(new_slice["Parea"] == new_slice["Parea"].max()) & 
                                               (new_slice["Parea"] > 0), "well_no"])
                
                if len(best_well_poss2) > 0:
                    best_well = best_well_poss2[0]
                else:
                    best_well = -1

        else:
            best_well_poss = list(new_slice.loc[(new_slice["Parea"] == new_slice["Parea"].max()) & 
                                               (new_slice["Parea"] > 0), "well_no"])
            if len(best_well_poss) > 0:
                best_well = best_well_poss2[0]
            else:
                best_well = -1
            
        row["best_wellno"] = best_well
        row["best_well"] = getUserReadableWell(best_well, self.plate_col_no) if best_well > -1 else "None found."
        return row
    
    def setBestMS(self, row, lcData):
        
        best_well = row["best_wellno"]
        peakID = -1
        for i in row["final_result"]["green"]:
            if lcData.loc[i, "well"] == best_well:
                peakID = lcData.loc[i, "peakID"]
        
        MS_plus = "-"
        MS_minus = "-"
        if peakID > -1:
            for hit in row["hits"]:
                if hit["well"] == best_well and hit["peakID"] == peakID:
                    MS_plus = hit["MS_plus"]
                    MS_minus = hit["MS_minus"]
        row["MS_plus"] = MS_plus
        row["MS_minus"] = MS_minus
        return row
        
        


In [478]:
#cpTable = Assignment("example_dataset/Waters/Example2/PyParse_designer_platemap.csv", 12)
cpTable = Assignment("example_dataset/Waters/Example1/example_platemap.csv", 12)

In [479]:
cpTable.generateCPTable()

In [480]:
cpTable.generateEMs("True")

In [481]:
msData = test.rawMSTable
uvData = test.rawUVTable
detector = "UV"

if detector == "UV":  
    lcData = test.rawDADTable
else:
    lcData = test.rawELSDTable

In [482]:
cpTable.findHits(msData)

In [483]:
cpTable.validateHits(lcData, msData, uvData)

In [484]:
class Output:
    def __init__(self, sample_IDs, plate_col_no, plate_row_no):
        #self.df = DataFrame[columns = well_no, Well, "SMILES", "SMarea", "S"]
        self.plate_col_no = plate_col_no
        self.plate_row_no = plate_row_no
        self.sample_IDs = sample_IDs
        
        #Insert blanks in sample_IDs to match length of datatable
        for i in range(1, plate_row_no * plate_col_no +1):
            if i not in self.sample_IDs:
                self.sample_IDs[i] = "No Data."
        
    def generateOutputTable(self, cpTable, lcData):
        """
        Reformats the validated hits into a pandas table ready for visualisation and export. 

        :param compoundDF: The pandas datatable containing all compounds with their respective hits.
        :param internalSTD: the name of the internalSTD
        :param SMs: a list of indices for the starting materials
        :param products: a list of indices for the products
        :param by_products: a list of indices for the by-products
        :param total_area_abs: A float corresponding to the sum of all peak_area_absolutes

        :return: A Pandas table named outputTable

        """
        
        def addCompoundToTable(row):
            #Add the product SMILES to the outputTable
            if row["type"] == "Product":
                for location in row["locations"]:
                    outputTable[location]["canonSMILES"] = row["smiles"]


            if len(row["final_result"]["green"]) != 0:
                max_area = max([lcData.loc[i, "area"] for i in row["final_result"]["green"]])

                for i in row["final_result"]["green"]:
                    if row["type"] == "Reactant":
                        outputTable[lcData.loc[i, "well"]]["SMarea"] = lcData.loc[i, "area"]
                        outputTable[lcData.loc[i, "well"]]["SMareaAbs"] = lcData.loc[i, "areaAbs"]
                        outputTable[lcData.loc[i, "well"]]["corrSMarea"] = lcData.loc[i, "area"] / max_area
                        outputTable[lcData.loc[i, "well"]]["Uarea"] = outputTable[lcData.loc[i, "well"]]["Uarea"] - lcData.loc[i, "area"]
                        outputTable[lcData.loc[i, "well"]]["UareaAbs"] = outputTable[lcData.loc[i, "well"]]["UareaAbs"] - lcData.loc[i, "areaAbs"]
                    
                    elif row["type"] == "InternalSTD":
                        outputTable[lcData.loc[i, "well"]]["STDarea"] = lcData.loc[i, "area"]
                        outputTable[lcData.loc[i, "well"]]["STDareaAbs"] = lcData.loc[i, "areaAbs"]
                        outputTable[lcData.loc[i, "well"]]["corrSTDarea"] = lcData.loc[i, "area"] / max_area
                        outputTable[lcData.loc[i, "well"]]["Uarea"] = outputTable[lcData.loc[i, "well"]]["Uarea"] - lcData.loc[i, "area"]
                        outputTable[lcData.loc[i, "well"]]["UareaAbs"] = outputTable[lcData.loc[i, "well"]]["UareaAbs"] - lcData.loc[i, "areaAbs"]
                    
                    elif row["type"] == "Product":
                        outputTable[lcData.loc[i, "well"]]["Parea"] = lcData.loc[i, "area"]
                        outputTable[lcData.loc[i, "well"]]["PareaAbs"] = lcData.loc[i, "areaAbs"]
                        outputTable[lcData.loc[i, "well"]]["corrParea"] = lcData.loc[i, "area"] / max_area
                        outputTable[lcData.loc[i, "well"]]["Uarea"] = outputTable[lcData.loc[i, "well"]]["Uarea"] - lcData.loc[i, "area"]
                        outputTable[lcData.loc[i, "well"]]["UareaAbs"] = outputTable[lcData.loc[i, "well"]]["UareaAbs"] - lcData.loc[i, "areaAbs"]
                        
                    elif row["name"] in by_products:
                        name = f'{row["name"]}area'
                        corr_name = f'corr{row["name"]}area'
                        abs_name = f'{row["name"]}areaAbs'
                        outputTable[lcData.loc[i, "well"]][name] = lcData.loc[i, "area"]
                        outputTable[lcData.loc[i, "well"]][abs_name] = lcData.loc[i, "areaAbs"]
                        outputTable[lcData.loc[i, "well"]][corr_name] = lcData.loc[i, "area"] / max_area
                        outputTable[lcData.loc[i, "well"]]["Uarea"] = outputTable[lcData.loc[i, "well"]]["Uarea"] - lcData.loc[i, "area"]
                        outputTable[lcData.loc[i, "well"]]["UareaAbs"] = outputTable[lcData.loc[i, "well"]]["UareaAbs"] - lcData.loc[i, "areaAbs"]
        
        def calculateNormalisedMetrics(row):
            #Calculate the corrP/STD value for each row
            
            max_P_STD = self.df.loc[self.df["canonSMILES"] == row["canonSMILES"], "P/STD"].max()
            if max_P_STD != 0:
                row["corrP/STD"] = row["P/STD"]/ max_P_STD
            else:
                row["corrP/STD"] = 0
            
            #Calculate the corrP/SM+P value for each row
            max_P_SM = self.df.loc[self.df["canonSMILES"] == row["canonSMILES"], "P/SM+P"].max()
            if max_P_SM != 0:
                row["corrP/SM+P"] = row["P/SM+P"]/ max_P_SM
            else:
                row["corrP/SM+P"] = 0
            
            return row
                                                         
        
        outputTable = {}
        
        products = list(cpTable.loc[cpTable["type"] == "Product", "name"])
        internalSTD = list(cpTable.loc[cpTable["type"] == "InternalSTD", "name"])
        SMs = list(cpTable.loc[cpTable["type"] == "Reactant", "name"])
        byproducts = list(cpTable.loc[cpTable["type"] == "Byproduct", "name"])
        #generate the structure of the table so that it is independant of the 
        #hits that are found

        for i in range(1, self.plate_row_no * self.plate_col_no + 1):
            well_id = getUserReadableWell(i, self.plate_col_no)
            total_area_abs = lcData.loc[lcData["well"] == i][["areaAbs"]].sum().values[0]

            goingIn = {
                "well_no": i,
                "Well": well_id,
                "SMILES": "",
                "canonSMILES": "",
                "SMarea": 0,
                "SMareaAbs": 0,
                "Parea": 0,
                "PareaAbs": 0,
                "STDarea": 0,
                "STDareaAbs": 0,
                "Uarea": 100,
                "UareaAbs": total_area_abs,
                "corrSMarea": 0,
                "corrParea": 0,
                "corrSTDarea": 0,
                "P/SM+P": 0,
                "P/STD":0
                }
            
            
            for by_prod in byproducts:            
                name = f'{by_prod}area'
                corr_name = f'corr{by_prod}area'
                abs_name = f'{by_prod}areaAbs'
                name_std = f'{by_prod}/STD'

                goingIn[name] = 0
                goingIn[abs_name] = 0
                goingIn[corr_name] = 0
                goingIn[name_std] = 0   

            outputTable[i] = goingIn  

        #Fill in the output table using the given cpTable and lcData
        cpTable.apply(addCompoundToTable, axis = 1)
        
        #reset any non-sensical values that arise as a result of miniscule rounding errors
        #for the unidentified area (Uarea, UareaAbs)
        for i in range(1, self.plate_row_no * self.plate_col_no + 1):

            well = outputTable[i]
            
            if round(well["Uarea"]) <= 0 or round(well["UareaAbs"]) <= 0: 
                well["Uarea"] = 0
                well["UareaAbs"] = 0

        #generate pandas dataframe, and sort it on the well_no (index)
        self.df = pd.DataFrame.from_dict(outputTable, orient="index")
        self.df.sort_index(inplace=True)
        
        #calculate the hybrid metrics
        self.df["P/SM+P"] = self.df.apply(lambda row: round(row["PareaAbs"] / (row["SMareaAbs"] + row["PareaAbs"]), 2) 
                                             if row["PareaAbs"] != 0 else 0, axis = 1)
        
        self.df["P/STD"] = self.df.apply(lambda row: round(row["PareaAbs"] / row["STDareaAbs"], 2) 
                                             if row["STDareaAbs"] != 0 else 0, axis = 1)
        for byprod in byproducts:
            name_std = f'{by_prod}/STD'
            name = f'{by_prod}areaAbs'
            self.df[name_std] = self.df.apply(lambda row: round(row[name] / row["STDareaAbs"], 2) 
                                             if row["STDareaAbs"] != 0 else 0, axis = 1)
            
        #calculate the normalised metrics
        
        self.df = self.df.apply(calculateNormalisedMetrics, axis = 1)
            
        
        
        

        

In [485]:
output = Output(test.sample_IDs, 12, 2)

In [486]:
output.generateOutputTable(cpTable.cpTable, lcData)

In [487]:
cpTable.cpTable = cpTable.cpTable.apply(cpTable.setBestWell, args=(output.df, "Parea",), axis = 1)
cpTable.cpTable = cpTable.cpTable.apply(cpTable.setBestMS, args=(lcData,), axis = 1)

In [488]:
cpTable.cpTable

,smiles,type,locations,name,rt,comments,mass1,mass2,mass3,hits,clusters,cluster_bands,refined_clusters,discarded_clusters,clusters_indexed_by_well,final_result,best_wellno,best_well,MS_plus,MS_minus
Brc1ccc(Br)c2ncccc12,Brc1ccc(Br)c2ncccc12,Product,[20],Product1,0,[Validation was not performed for Product1 as ...,284.88,286.88,0.00,"[{'well': 20.0, 'peakID': 1.0, 'mass_conf': 68...",[[28]],[1.1763],"[{'green': [28], 'orange': [], 'discarded': []}]",[],"{20: {'green': [28], 'orange': [], 'discarded'...","{'green': [28], 'discarded': []}",20,B8,287.86,-
Brc1ccc2cccc(Br)c2n1,Brc1ccc2cccc(Br)c2n1,Product,[5],Product2,0,[Validation was not performed for Product2 as ...,284.88,286.88,0.00,"[{'well': 5.0, 'peakID': 1.0, 'mass_conf': 73....",[[6]],[1.2246],"[{'green': [6], 'orange': [], 'discarded': []}]",[],"{5: {'green': [6], 'orange': [], 'discarded': ...","{'green': [6], 'discarded': []}",5,A5,287.84,-
Brc1cccnc1N1CCOCC1,Brc1cccnc1N1CCOCC1,Product,[21],Product3,0,[Validation was not performed for Product3 as ...,242.01,244.01,0.00,"[{'well': 21.0, 'peakID': 1.0, 'mass_conf': 94...",[[30]],[0.9542],"[{'green': [30], 'orange': [], 'discarded': []}]",[],"{21: {'green': [30], 'orange': [], 'discarded'...","{'green': [30], 'discarded': []}",21,B9,245.05,-
Brc1cnc2ccccc2c1,Brc1cnc2ccccc2c1,Product,[16],Product4,0,[Validation was not performed for Product4 as ...,206.97,208.97,0.00,"[{'well': 16.0, 'peakID': 1.0, 'mass_conf': 77...",[[22]],[1.0654],"[{'green': [22], 'orange': [], 'discarded': []}]",[],"{16: {'green': [22], 'orange': [], 'discarded'...","{'green': [22], 'discarded': []}",16,B4,208.0,-
CC(C)(C)OC(=O)N1CCN(C(=O)OCC2c3ccccc3-c3ccccc32)CC1C(=O)O,CC(C)(C)OC(=O)N1CCN(C(=O)OCC2c3ccccc3-c3ccccc3...,Product,[17],Product5,0,[Validation was not performed for Product5 as ...,452.19,396.13,352.14,"[{'well': 17.0, 'peakID': 1.0, 'mass_conf': 38...",[[23]],[0.9017],"[{'green': [23], 'orange': [], 'discarded': []}]",[],"{17: {'green': [23], 'orange': [], 'discarded'...","{'green': [23], 'discarded': []}",17,B5,397.23,-
CC(C)(C)OC(=O)N1CCN(c2ccc(Br)cn2)CC1,CC(C)(C)OC(=O)N1CCN(c2ccc(Br)cn2)CC1,Product,[13],Product6,0,[Validation was not performed for Product6 as ...,341.07,285.01,241.02,"[{'well': 13.0, 'peakID': 1.0, 'mass_conf': 44...","[[17], [18]]","[1.3217, 1.4329]","[{'green': [17], 'orange': [], 'discarded': []...",[],"{13: {'green': [17, 18], 'orange': [], 'discar...","{'green': [17], 'discarded': [18]}",13,B1,342.09,-
CC(C)(C)OC(=O)N1CCN(c2ccnc(Cl)n2)CC1,CC(C)(C)OC(=O)N1CCN(c2ccnc(Cl)n2)CC1,Product,[18],Product7,0,[Validation was not performed for Product7 as ...,298.12,242.06,198.07,"[{'well': 18.0, 'peakID': 1.0, 'mass_conf': 64...","[[25], [26]]","[1.0684, 1.2758]","[{'green': [25], 'orange': [], 'discarded': []...",[],"{18: {'green': [25, 26], 'orange': [], 'discar...","{'green': [26], 'discarded': [25]}",18,B6,299.12,-
CC(C)(C)OC(=O)N1CCN(c2ccnc3[nH]ccc23)CC1,CC(C)(C)OC(=O)N1CCN(c2ccnc3[nH]ccc23)CC1,Product,[14],Product8,0,[Validation was not performed for Product8 as ...,302.17,246.11,202.12,"[{'well': 14.0, 'peakID': 2.0, 'mass_conf': 86...",[[20]],[1.0396],"[{'green': [20], 'orange': [], 'discarded': []}]",[],"{14: {'green': [20], 'orange': [], 'discarded'...","{'green': [20], 'discarded': []}",14,B2,303.19,-
CC(C)(C)OC(=O)N1CCN(c2ncc(Br)cn2)CC1,CC(C)(C)OC(=O)N1CCN(c2ncc(Br)cn2)CC1,Product,[15],Product9,0,[Validation was not performed for Product9 as ...,342.07,286.01,242.02,"[{'well': 15.0, 'peakID': 1.0, 'mass_conf': 45...",[[21]],[1.3221],"[{'green': [21], 'orange': [], 'discarded': []}]",[],"{15: {'green': [21], 'orange': [], 'discarded'...","{'green': [21], 'discarded': []}",15,B3,287.0,-
CCc1ccc2cc(C(=O)O)cnc2c1,CCc1ccc2cc(C(=O)O)cnc2c1,Product,[3],Product10,0,[Validation was not performed for Product10 as...,201.08,0.00,0.00,"[{'well': 3.0, 'peakID': 1.0, 'mass_conf': 156...",[[3]],[0.5863],"[{'green': [3], 'orange': [], 'discarded': []}]",[],"{3: {'green': [3], 'orange': [], 'discarded': ...","{'green': [3], 'disca

In [237]:
def plotPieCharts(zvalue, outputTable, save_dir, by_products, plate_row_no, plate_col_no):
    """
    Plots a set of pie charts for the full plate
    using the full dataset. The size of the pie chart is dependant on
    the value in the datatable for the column specified by zvalue
    
    :param zvalue: a string corresponding to the desired output metric (e.g. P/STD)
    :param outputTable: a pandas datatable
    :param save_dir: a string for the output directory
    :param by_products: a list of names of byproducts
    
    :return: jpg of the piecharts saved to output directory
    """

    #declare color palette
    palette = ["black", "brown", "red", "sienna",
                "peru", "orange", "gold", "olive", "lawngreen", "darkgreen", "lime", "aqua", 
                "steelblue", "slategray", "navy"]
    
    
    def buildPies(chart_type):
        """
        Define function to build a trellised pie chart. 

        :param chart_type: String defining whether this set of pie charts has fixed or variable 
                            diameters
        
        :return: jpg of trellised pie chart saved to output folder
        """

        #declare new subplots
        fig, axs = plt.subplots(plate_row_no, plate_col_no)
        
        #get maximum value, so diameter of all pies can be calculated in relation
        #to the best performing well. 
        max_val = outputTable[zvalue].max()
        
        #format data and generate pie charts
        pies_baked = []
        if max_val != 0:
            
            for index, row in outputTable.iterrows():
                pies_baked.append(index)
                rowVal = math.floor((index-1) / plate_col_no)
                colVal = int((index-1) % plate_col_no)

                if chart_type == "fixed_width":
                    chart_size = 0.95
                else:
                    chart_size =  (row[zvalue] / max_val) * 0.95
                    print(row[zvalue], max_val)

                color_tracker = 0
                colors = []
                sum_byprod = sum([row[f'{by_prod}area'] for by_prod in by_products])
                
                total = row["SMarea"] + row["Parea"] + row["STDarea"] + sum_byprod
                #the LCMS machine can calculate a total area of >100% due to rounding errors
                #this needs to be corrected before the piechart is built
                if total > 100:
                    total = 100
                data = []
                if chart_size > 0: 
                    if total != 100:
                        data.append(100-total)
                        colors.append("grey")

                    if row["SMarea"] != 0:
                        data.append(math.floor(row["SMarea"]))
                        colors.append("cyan")

                    if row["Parea"] != 0:
                        data.append(math.floor(row["Parea"]))
                        colors.append("yellow")

                    if row["STDarea"] != 0:
                        data.append(math.floor(row["STDarea"]))
                        colors.append("green")
                    
                    for by_prod in by_products:
                        if math.floor(row[f'{by_prod}area']) != 0:
                            data.append(math.floor(row[f'{by_prod}area']))
                            colors.append(palette[color_tracker])
                        color_tracker = color_tracker + 1
                else:
                    #Set chart size to 0.01 to avoid a non-zero radius
                    #which causes matplotlib to fail. 
                    chart_size = 0.01
                    
                
                if plate_row_no == 1:
                    axs[colVal].pie(data, 
                            textprops={"size": "smaller"}, 
                            colors = colors,  
                            radius=chart_size,
                            normalize = True)
                elif plate_col_no == 1:
                    axs[rowVal].pie(data, 
                            textprops={"size": "smaller"}, 
                            colors = colors,  
                            radius=chart_size,
                            normalize = True)
                else:
                    axs[rowVal, colVal].pie(data, 
                            textprops={"size": "smaller"}, 
                            colors = colors,  
                            radius=chart_size,
                            normalize = True)
                
        #Remove any unused sections of the trellised pie chart
        #graph so that the visualisation is neater.         
        for i in range(1, plate_col_no * (plate_row_no) + 1):
            rowVal = math.floor((i-1) / plate_col_no)
            colVal = int((i-1) % plate_col_no)
            if i not in pies_baked:
                
                #matplotlib axes drop a dimension 
                #when that dimension is length = 1. 
                #Adjust code to match
                if plate_row_no == 1:
                    plt.delaxes(axs[colVal])
                elif plate_col_no == 1:
                    plt.delaxes(axs[rowVal])
                else:
                    plt.delaxes(axs[rowVal, colVal])
           

        #Add a key to the graph 
        lines = [Line2D([0], [0], color="grey", lw=4), 
                Line2D([0], [0], color="cyan", lw=4),
                Line2D([0], [0], color="yellow", lw=4),
                Line2D([0], [0], color="green", lw=4),]
        labels = ["Untagged", "Reactant", "Product", "InternalSTD"]
        for i in range(len(by_products)):
            lines.append(Line2D([0], [0], color=palette[i], lw=4))
            labels.append(by_products[i])
        if plate_row_no == 1:
            axs[plate_col_no-1].legend(lines,
                    labels, loc="lower center",
                    bbox_to_anchor=(1,1), ncol=math.ceil(len(labels)/3))
        elif plate_col_no == 1:
            axs[plate_row_no-1].legend(lines,
                    labels, loc="upper left",
                    bbox_to_anchor=(1,1), ncol=math.ceil(len(labels)/3))
        else:
            axs[plate_row_no-1, plate_col_no-1].legend(lines,
                    labels, loc="upper right",
                    bbox_to_anchor=(1,0), ncol=math.ceil(len(labels)/3))
            
        #Add titles to the graphs
        if chart_type == "fixed_width":
            fig.suptitle("Fixed Diameter Trellised Pie Charts", y=0.9)
        else:
            fig.suptitle(f'Trellised Pie Charts Sized by {zvalue}', y=0.9)
        
        #Save graph to output directory
        plt.savefig(f'{save_dir}graphs/piecharts_{chart_type}.jpg', format="jpg")
        plt.close()
    
    buildPies("fixed_width")
    buildPies("variable_width")

In [239]:
def plotHeatmaps(outputTable, save_dir, plate_row_no, plate_col_no):
    """
    Plots and saves heatmaps for the full dataset
    
    :param outputTable: a pandas datatable
    :param save_dir: a string for the output directory
    
    :return: jpg of the heatmap saved to output directory
    """
    
    zvalues = {
        "SMarea": "SMarea",
        "Parea": "Parea", 
        "conversion": "P/SM+P", 
        "ratio_to_IS": "P/STD",
        "corrSMarea": "corrSMarea",
        "corrParea": "corrParea", 
        "corrected_conversion": "corrP/SM+P",
        "corrected_ratio_to_IS": "corrP/STD"
    }
    for key, zvalue in zvalues.items():
        
        returnVal = []
        labels = []
        #convert output table to 2-D list that can be used to plot heatmap. 
        for index, row in outputTable.iterrows():
            rowVal = int(math.floor((index-1) / plate_col_no))
            colVal = int((index-1) % plate_col_no)
            if colVal == 0:
                returnVal.append([])
                labels.append([])
            if row["SampleID"] == "No Data.":
                labels[rowVal].append("-")
                returnVal[rowVal].append(0)
            else:
                if row[zvalue] > 1:
                    labels[rowVal].append(math.floor(row[zvalue]))
                elif row[zvalue] >= 100:
                    labels[rowVal].append(100)
                elif row[zvalue] == 0:
                    labels[rowVal].append("0")
                else:
                    labels[rowVal].append(round(row[zvalue],2))

                returnVal[rowVal].append(row[zvalue])

        #Configure heatmap
        pdTable = pd.DataFrame(returnVal)
        xLabels = [i for i in range(1, plate_col_no+1)]
        yLabels = [chr(ord('@')+i) for i in range(1, plate_row_no+1)]
        
        ax = sns.heatmap(pdTable, xticklabels=xLabels, yticklabels=yLabels, cmap = "viridis",
                        annot = labels, cbar_kws={"label": zvalue}, fmt="")
        ax.xaxis.set_ticks_position("top")
        
        #Save heatmap to output directory
        plt.savefig(f'{save_dir}graphs/heatmap_{key}.jpg', format="jpg")
        plt.close()

In [ ]:
def plotChroma(row, outputTable, cpTable, lcData, msData, save_dir, plot_type):
    
    #cpname, wellno, trace, pStart, pEnd, annotate_peaks, save_dir, 
    #           ms_plus, ms_minus, mass1):
    """
    Plots the LCMS trace with labels for compounds found for a specific well,
    and highlights a particular peak of interest, providing m/z data for that
    peak. 

    :param cpname: a string of the compound name
    :param wellno: an integer representing the well
    :param trace: a list [x-values, y-values] to plot the Uv chromatogram 
    :param pStart: a float for the time where a specific peak begins
    :param pEnd: a float for the time where a specific peak ends. 
    :param annotate_peaks: a list of dictionaries for peaks to annotate
    :param save_dir: a string for the output directory
    :param ms_plus: a list [x-values, y-values] for MS+ spectrometric data
    :param ms_minus: a list [x-values, y-values] for MS- spectrometric data
    :param mass1: the isotopic mass of a compound, to which +1/-1 should be added
        to get to an expected observed mass (typically parent isotopic mass)
    
    :return: jpg of the chromatogram saved to output directory
    """
    
    
    #Function to plot the mass spectrometric data
    def plotMS(axes, well, peakID, title):
        x = list(msData.loc[(msData["well"] == well) & (msData["peakID"] == peakID), "MSvalue"])
        y = list(msData.loc[(msData["well"] == well) & (msData["peakID"] == peakID), "MSintensity"])
        
        axes.bar(x, y, width = 2)
        axes.set_title(title)
        axes.set_ylim(0, 200)
        
        last_annotation = 0
        highest_mass = 0
        for i, j in enumerate(x):
            if y[i] > 20:
                if math.isclose(j, last_annotation, abs_tol = 3):
                    axes.annotate(j, [j+15, y[i]+20], ha="center", 
                                  va="bottom", rotation=90, size = 8)
                    axes.arrow(j+10, y[i]+18, -10, -8)
                else:
                    axes.annotate(j, [j+1, y[i]], ha="center", 
                                  va="bottom", rotation=90, size = 8)
                last_annotation = j
                if j > highest_mass:
                    highest_mass = j
        
        #Configure mass spec axis to reach at least 1000, but higher if necessary
        #With some additional headroom so that peaks are not obscured by the edge
        #of the plot
        if highest_mass > 950:
            axes.set_xlim(0, math.ceil((highest_mass + 100)/100)*100) 
        else:
            axes.set_xlim(0, 1000)
    
    fig, (a0, a1, a2) = plt.subplots(3, 1, gridspec_kw={'height_ratios': [2, 2, 6]})
    
    #First, determine for the compound given which well it is that should be plotted
    
    if len(row["final_result"]["green"]) > 0:
        well_to_use = row["best_wellno"]
    elif len(row["final_result"]["discarded"]) == 1:
        well_to_use = lcData.loc[row["final_result"]["discarded"][0], "well"]
    elif len(row["locations"]) == 1:
        well_to_use = row["locations"][0]
        
    annotate_peaks = []
    for bindex, brow in compoundDF.iterrows():
        data = [i for i in brow["hits"]["green"] if i["well"] == only_well]
        if len(data) > 0:
            annotate_peaks.append({
                    "cpname": brow["name"], 
                    "time": data[0]["time"]
                    })
    
    
    #Plot both MS- and MS+    
    plotMS(a0, ms_minus, "MS-")
    plotMS(a1, ms_plus, "MS+")

    #Plot the UV chromatogram
    a2.plot(trace[0], trace[1])

    #label the graph and axes
    label = getUserReadableWell(wellno)
    fig.suptitle(f'{cpname} ({mass1}): Well {label}')
    a2.set_xlabel("Time /min")
    a2.set_ylabel("AUs")

    #Find the index of the start and end times of the peak
    #that should be highlighted. 
    if pStart != 0 and pEnd != 0:
        x_index_start = min(enumerate(trace[0]), 
                        key = lambda x: abs(x[1]-pStart))[0]
        x_index_end = min(enumerate(trace[0]), 
                        key = lambda x: abs(x[1]-pEnd))[0]

        diff = trace[1][x_index_end] - trace[1][x_index_start]
        
        #Generate a second curve which can be used to specifically fill the hit
        #well. 
        second_curve = []
        for index, value in enumerate(trace[0]):
            #Calculate where the baseline value should be based on the relative
            #heights at the start and end of the peak. 
            if index >= x_index_start and index <= x_index_end:
                new_val = (trace[1][x_index_start] 
                    + diff*((index - x_index_start)/(x_index_end - x_index_start)))
                if new_val > trace[1][index]:
                    second_curve.append(trace[1][index])
                else:
                    second_curve.append(new_val)
            #For all x-values that don't fill inside the hit peak, 
            #the second curve should match the LCMS trace so that these areas don't
            #get filled. 
            else:
                second_curve.append(trace[1][index])
            
    #Annotate the peaks that were matched to a compound using an arrow
    #and dynamic positioning for clarity
    annotate_peaks.sort(key = lambda x: x["time"])
    last_x_position = 0
    max_x = max(trace[0])
    for index, i in enumerate(annotate_peaks):

        x_index = min(enumerate(trace[0]), 
                      key = lambda x: abs(x[1]-i["time"]))[0]
        if index < len(annotate_peaks) / 2:
            h_align = "right"
        else:
            h_align = "left"
        if index == 0:
            offset = -(max_x / 40)
        elif index == len(annotate_peaks)-1:
            offset = (max_x / 40)
        else:
            offset = 0
        arrow_props = dict(arrowstyle="->", connectionstyle="arc3")
        if math.isclose(last_x_position, i["time"]+offset, abs_tol = (max_x / 20)) and index != len(annotate_peaks)-1:
            if h_align == "left":
                h_align = "right"
            else:
                h_align = "left"

        if index % 2 == 0:
            plt.annotate(f'{i["cpname"]}: {i["time"]}', xy = (i["time"], trace[1][x_index]), 
                        xytext = (i["time"]+offset, 120), horizontalalignment=h_align,
                        arrowprops = arrow_props)
        else:
            plt.annotate(f'{i["cpname"]}: {i["time"]}', xy = (i["time"], trace[1][x_index]), 
                        xytext = (i["time"]+offset, 110), horizontalalignment=h_align,
                        arrowprops = arrow_props)
        last_x_position = i["time"]+offset
    min_y = min(trace[1])
    
    #Set axis limits
    a2.set_ylim(min_y - 10, 130)
    a2.set_xlim(-0.1, max_x+0.1)

    #plot a hatched region if the filter_by_rt option 
    #has been used
    if len(options.filter_by_rt) > 0:
        
        #Set default yvalues to match that of chromatogram trace
        hatched_ymin, hatched_ymax = [],[]
        for i in trace[1]:
            hatched_ymin.append(i)
            hatched_ymax.append(i)

        for new_range in options.filter_by_rt:
            minx = float(new_range.split("-")[0].strip())
            maxx = float(new_range.split("-")[1].strip())
            for i, j in enumerate(trace[0]):
                #Amend any values such between specified ranges
                if j > minx and j < maxx:
                    hatched_ymax[i] = 130
                    hatched_ymin[i] = min_y - 10
        #fill between on graph, using traceparency = 0.3
        a2.fill_between(trace[0], hatched_ymin, hatched_ymax, hatch = "/", alpha = 0.3)

    #Fill to highlight the peak of interest 
    if pStart != 0 and pEnd != 0:
        a2.fill_between(trace[0], second_curve, trace[1], color="red")

    #Set the number of tick points for each axis. 
    a2.locator_params(axis='y', nbins=6)
    a2.locator_params(axis='x', nbins=20)
    
    #Save graph to output directory. 
    plt.savefig(f'{save_dir}graphs/chroma-{cpname}-best.jpg', format="jpg")
    plt.close()

In [240]:
save_dir = "example_dataset/Waters/Example1/code_reop_output/"
byproducts = list(cpTable.cpTable.loc[cpTable.cpTable["type"] == "Byproduct", "name"])
print(byproducts)
plotHeatmaps(output.df, save_dir, 2, 12)

[]


KeyError: 'SampleID'